# Demo - dbt Projects on Snowflake: bring your existing dbt project

**Why we are doing this.** Your team already writes dbt. You can keep it. dbt runs **inside Snowflake**: no dbt Cloud, no separate server, no extra credentials to manage.

**For the Databricks and ADF engineers:** today dbt runs somewhere outside the warehouse. Here, the project is an object in Snowflake. You run it with one SQL command, and you schedule it with a task.

**Where everything lives**
| What | Where |
|---|---|
| The dbt files (models, tests) | This workspace, folder **somatus_dbt** |
| The deployed dbt project | **SOMATUS_DEMO.OPS.SOMATUS_DBT**, an object in a schema, like a table |
| The table it builds | **SOMATUS_DEMO.GOLD.FCT_CLAIMS** |

**Labels:** **SETUP** (already built, read only) · **RUN DURING DEMO** · **UI STEP** · **SAY THIS**

### RUN DURING DEMO
Set the role, database and warehouse. Run this first.

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE SOMATUS_DEMO;
USE WAREHOUSE WH_TRANSFORM;
USE SCHEMA OPS;

### UI STEP - do this in the screen, not in the notebook
**Show the dbt files**

1. In the workspace file list on the left, open **somatus_dbt > models > fct_claims.sql**. It is an incremental model: each run processes only claims loaded since the last run.
2. Open **schema.yml**. These are the tests (claim_id not null and unique, member_id and paid_amount not null). They run on every build.
3. Come back to this notebook.

### RUN DURING DEMO
The deployed dbt project, an object in the OPS schema.

In [ ]:
%%sql -r dbt_project
SHOW DBT PROJECTS IN SCHEMA SOMATUS_DEMO.OPS;

> **SAY THIS:** Your dbt project, unchanged, running inside Snowflake. It's incremental, so each run processes only new claims, and the tests run on every build. To run it on a schedule, a task runs one line: EXECUTE DBT PROJECT.

### IF ASKED
- **"How do you run it?"** `EXECUTE DBT PROJECT SOMATUS_DEMO.OPS.SOMATUS_DBT ARGS = 'build';` from SQL, a task, or `snow dbt execute` from CI.
- **"Can dbt build a dynamic table?"** Yes: `materialized = 'dynamic_table'` with a `target_lag`.
- **"Can we keep using Git?"** Yes. A workspace can be connected to your Git repository, and a project can be deployed from Git.

## What we just proved
1. **Your dbt project runs inside Snowflake, unchanged.** No dbt Cloud, no separate server.
2. **It is incremental.** Each run processes only the claims loaded since the last run.
3. **Tests run on every build.** Bad data stops before it reaches Gold.
4. **It is one object in a schema.** Run it with EXECUTE DBT PROJECT, schedule it with a task, deploy it from Git.